# EP2 — AVY1103 Deep Learning: Reconocimiento Facial (CNN + Dataset GitHub)
## Entrenamiento de 12 configuraciones CNN conectadas a `The-Nuk/proyecto-dataset-vision`

Este cuaderno realiza el flujo completo de visión artificial:
1. Clona el repositorio colaborativo `https://github.com/The-Nuk/proyecto-dataset-vision.git`.
2. **Unifica automáticamente siglas y nombres reales** (ej. `mf 01` y `Matías Figueroa` se combinan; `ci 01` y `compañero inf` se separan como personas desconocidas para calibrar el umbral).
3. Realiza una **división estratificada segura** (train/val/test) garantizando que no falle aunque haya alumnos con pocas fotos.
4. Entrena 12 configuraciones de Redes Neuronales Convolucionales (CNN) con Data Augmentation, Batch Normalization y Dropout.
5. Calibra el umbral de confianza para detectar personas ajenas al dataset.
6. Genera y descarga automáticamente `modelo_cnn.keras` y `clases.json` listos para la API (FastAPI) y la app móvil (Android e iPhone).

**Antes de correr:**
- En Google Colab: Menú `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → Selecciona **GPU T4**.

In [ ]:
from google.colab import drive, files
import os, sys, json, time, shutil, csv, unicodedata
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Montar Google Drive (opcional: para respaldos o si tus fotos están en Drive)
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive no conectado (opcional):", e)

SEMILLA = 42
TAMANO_IMAGEN = 128
EPOCAS_MAXIMAS = 40
TAMANO_LOTE = 16
RUTA_SALIDA = "/content/resultados"
os.makedirs(f"{RUTA_SALIDA}/curvas", exist_ok=True)
os.makedirs(f"{RUTA_SALIDA}/modelos", exist_ok=True)
np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

## 0. Clonar Dataset de GitHub (`The-Nuk/proyecto-dataset-vision`)
Clona automáticamente el repositorio que contiene las imágenes y la estructura del proyecto.

In [ ]:
# 1. Clonar el repositorio si aún no está en el entorno
REPO_DIR = "/content/proyecto-dataset-vision"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/The-Nuk/proyecto-dataset-vision.git {REPO_DIR}
    print("Repositorio clonado exitosamente en:", REPO_DIR)
else:
    print("El repositorio ya existe en:", REPO_DIR)

# 2. Rutas del dataset dentro del repositorio
RUTA_IMAGENES_REPO = os.path.join(REPO_DIR, "data", "images")
RUTA_SUJETOS_CSV = os.path.join(REPO_DIR, "sujetos.csv")
RUTA_DESCONOCIDOS_DRIVE = "/content/drive/MyDrive/desconocidos"  # opcional

# 3. Detección automática de la carpeta con imágenes
RUTA_DRIVE_DATASET = "/content/drive/MyDrive/dataset"
if os.path.exists(RUTA_IMAGENES_REPO) and len(os.listdir(RUTA_IMAGENES_REPO)) > 0:
    RUTA_DATASET = RUTA_IMAGENES_REPO
    print("Usando imágenes del repositorio clonado:", RUTA_DATASET)
elif os.path.exists(RUTA_DRIVE_DATASET):
    RUTA_DATASET = RUTA_DRIVE_DATASET
    print("Usando imágenes desde Google Drive:", RUTA_DATASET)
else:
    RUTA_DATASET = RUTA_IMAGENES_REPO
    print("Ruta asignada:", RUTA_DATASET)

## 1. Carga, Unificación de Alias y División Estratificada Segura

- Normaliza tildes y caracteres Unicode (NFC).
- Unifica siglas (ej. `mf 01` con `Matías Figueroa`, `cv 01` con `Carlos Vasquez`, `bp 01` con `Benjamín Palma`).
- Separa las fotos de `compañero inf` / `ci 01` para usarlas como **personas desconocidas** en la calibración del umbral.
- Aplica una división estratificada inteligente que no falla aunque haya clases con pocas fotos.

In [ ]:
import unicodedata, csv, cv2
import numpy as np, pandas as pd

def normalizar_texto(texto):
    """Normaliza a forma canónica NFC (unifica tildes y eñes)."""
    if not texto:
        return ""
    return unicodedata.normalize('NFC', str(texto)).strip()

# Mapeo de siglas y carpetas alternas a nombres completos
MAPA_ALIAS = {
    "mf 01": "Matías Figueroa",
    "mf_01": "Matías Figueroa",
    "cv 01": "Carlos Vasquez",
    "cv_01": "Carlos Vasquez",
    "jm 01": "Javier Montupil",
    "jm_01": "Javier Montupil",
    "la 01": "Luciano Alcantara",
    "la_01": "Luciano Alcantara",
    "bg 01": "Benjamín Gálvez",
    "bg_01": "Benjamín Gálvez",
    "bp 01": "Benjamín Palma",
    "bp_01": "Benjamín Palma",
    "ci 01": "compañero inf",
    "ci_01": "compañero inf",
}

def leer_imagen_rgb(ruta):
    imagen = cv2.imread(ruta)
    if imagen is None:
        return None
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    return cv2.resize(imagen, (TAMANO_IMAGEN, TAMANO_IMAGEN))

def cargar_dataset_unificado(ruta_dataset, ruta_csv):
    """Carga imágenes, unifica alias y separa fotos desconocidas."""
    mapa_csv = {}
    if ruta_csv and os.path.exists(ruta_csv):
        with open(ruta_csv, mode="r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for fila in reader:
                id_c = normalizar_texto(fila.get("id_carpeta", ""))
                nom = normalizar_texto(fila.get("nombre_original") or fila.get("nombre_normalizado", ""))
                if id_c and nom:
                    mapa_csv[id_c.lower()] = nom

    imagenes_conocidos, etiquetas_conocidos = [], []
    fotos_desconocidas = []

    if not os.path.exists(ruta_dataset):
        raise FileNotFoundError(f"No existe la carpeta de imágenes: {ruta_dataset}")

    for carpeta_id in sorted(os.listdir(ruta_dataset)):
        carpeta = os.path.join(ruta_dataset, carpeta_id)
        if not os.path.isdir(carpeta) or carpeta_id.startswith("."):
            continue

        c_norm = normalizar_texto(carpeta_id)
        c_lower = c_norm.lower()

        # Resolver nombre con sujetos.csv o diccionario de alias
        nombre_final = mapa_csv.get(c_lower, MAPA_ALIAS.get(c_lower, c_norm))
        nombre_final = normalizar_texto(nombre_final)

        # Verificar si es persona desconocida / excluida de la base de datos
        es_desconocido = any(p in c_lower or p in nombre_final.lower()
                             for p in ["compañero", "companero", "ci 01", "ci_01", "desconocid"])

        for archivo in os.listdir(carpeta):
            if archivo.lower().endswith((".jpg", ".jpeg", ".png")):
                img = leer_imagen_rgb(os.path.join(carpeta, archivo))
                if img is not None:
                    if es_desconocido:
                        fotos_desconocidas.append(img)
                    else:
                        imagenes_conocidos.append(img)
                        etiquetas_conocidos.append(nombre_final)

    return (np.array(imagenes_conocidos, dtype="uint8"),
            np.array(etiquetas_conocidos),
            np.array(fotos_desconocidas, dtype="uint8") if len(fotos_desconocidas) > 0 else np.array([]))

ruta_csv_efectiva = globals().get('RUTA_SUJETOS_CSV', '')
X, etiquetas_texto, fotos_desconocidas = cargar_dataset_unificado(RUTA_DATASET, ruta_csv_efectiva)
print(f"Imágenes de alumnos conocidos cargadas: {len(X)}")
print(f"Imágenes de desconocidos separadas (para umbral): {len(fotos_desconocidas)}")

conteo = pd.Series(etiquetas_texto).value_counts()
print("\nDistribución unificada de fotos por alumno:")
print(conteo)

codificador = LabelEncoder()
y = codificador.fit_transform(etiquetas_texto)
nombres_clases = list(codificador.classes_)
numero_clases = len(nombres_clases)

def division_estratificada_segura(X_data, y_data, val_ratio=0.15, test_ratio=0.15, random_seed=42):
    """Divide en train/val/test asegurando que toda clase esté representada sin causar errores."""
    rng = np.random.RandomState(random_seed)
    idx_train, idx_val, idx_test = [], [], []

    for clase_id in np.unique(y_data):
        indices_clase = np.where(y_data == clase_id)[0]
        rng.shuffle(indices_clase)
        n = len(indices_clase)

        if n <= 2:
            idx_train.append(indices_clase[0])
            idx_val.append(indices_clase[-1])
            idx_test.append(indices_clase[-1])
        elif n == 3:
            idx_train.append(indices_clase[0])
            idx_val.append(indices_clase[1])
            idx_test.append(indices_clase[2])
        else:
            n_val = max(1, int(round(n * val_ratio)))
            n_test = max(1, int(round(n * test_ratio)))
            if n_val + n_test >= n:
                n_val = 1
                n_test = 1
            n_train = n - n_val - n_test
            idx_train.extend(indices_clase[:n_train])
            idx_val.extend(indices_clase[n_train:n_train + n_val])
            idx_test.extend(indices_clase[n_train + n_val:])

    idx_tr = np.array(idx_train)
    idx_v = np.array(idx_val)
    idx_te = np.array(idx_test)

    return X_data[idx_tr], X_data[idx_v], X_data[idx_te], y_data[idx_tr], y_data[idx_v], y_data[idx_te]

X_entrenamiento, X_validacion, X_prueba, y_entrenamiento, y_validacion, y_prueba = division_estratificada_segura(
    X, y, val_ratio=0.15, test_ratio=0.15, random_seed=SEMILLA)

print(f"\nConjuntos listos: train={len(X_entrenamiento)} | val={len(X_validacion)} | test={len(X_prueba)}")
print(f"Clases en train: {len(np.unique(y_entrenamiento))} | val: {len(np.unique(y_validacion))} | test: {len(np.unique(y_prueba))}")

## 2. Modelo: normalización y data augmentation dentro del modelo

- `Rescaling(1/255)` normaliza los píxeles a [0, 1]. Al estar dentro del modelo, la API recibe la matriz cruda 0-255 y no hay desajuste entre entrenamiento e inferencia.
- Las capas de aumento (volteo, rotación, zoom, brillo) solo se activan al entrenar.
- 12 configuraciones = 3 arquitecturas (1, 2 y 3 bloques Conv+MaxPool) × 4 niveles de regularización.

In [ ]:
def construir_cnn(numero_bloques, usar_dropout, usar_batchnorm, usar_aumento):
    entrada = layers.Input(shape=(TAMANO_IMAGEN, TAMANO_IMAGEN, 3))
    x = entrada
    if usar_aumento:
        x = layers.RandomFlip("horizontal")(x)
        x = layers.RandomRotation(15 / 360)(x)            # ±15 grados
        x = layers.RandomZoom(0.1)(x)
        x = layers.RandomBrightness(0.2, value_range=(0, 255))(x)
    x = layers.Rescaling(1.0 / 255)(x)
    for filtros in [32, 64, 128][:numero_bloques]:
        x = layers.Conv2D(filtros, 3, padding="same")(x)
        if usar_batchnorm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    if usar_dropout:
        x = layers.Dropout(0.4)(x)
    salida = layers.Dense(numero_clases, activation="softmax")(x)
    return models.Model(entrada, salida)

ARQUITECTURAS = {"CNN-A": 1, "CNN-B": 2, "CNN-C": 3}
REGULARIZACIONES = {
    "Base":            dict(usar_dropout=False, usar_batchnorm=False, usar_aumento=False),
    "+Dropout":        dict(usar_dropout=True,  usar_batchnorm=False, usar_aumento=False),
    "+BN+Dropout":     dict(usar_dropout=True,  usar_batchnorm=True,  usar_aumento=False),
    "+Aug+BN+Dropout": dict(usar_dropout=True,  usar_batchnorm=True,  usar_aumento=True),
}
configuraciones = []
for nombre_arquitectura, numero_bloques in ARQUITECTURAS.items():
    for nombre_regularizacion, opciones in REGULARIZACIONES.items():
        configuraciones.append(dict(id=f"C{len(configuraciones) + 1}", arquitectura=nombre_arquitectura,
                                    regularizacion=nombre_regularizacion, numero_bloques=numero_bloques, **opciones))
pd.DataFrame(configuraciones)[["id", "arquitectura", "regularizacion"]]

## 3. Entrenamiento de las 12 configuraciones (EarlyStopping + ReduceLROnPlateau)

In [ ]:
resultados, historiales = [], {}
for cfg in configuraciones:
    tf.keras.backend.clear_session()
    modelo = construir_cnn(cfg["numero_bloques"], cfg["usar_dropout"], cfg["usar_batchnorm"], cfg["usar_aumento"])
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics="accuracy")
    control = [callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
               callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)]
    inicio = time.time()
    historia = modelo.fit(X_entrenamiento, y_entrenamiento, validation_data=(X_validacion, y_validacion),
                          epochs=EPOCAS_MAXIMAS, batch_size=TAMANO_LOTE, callbacks=control, verbose=0)
    h = historia.history
    mejor_epoca = int(np.argmin(h["val_loss"]))
    y_pred_val = np.argmax(modelo.predict(X_validacion, verbose=0), axis=1)
    resultados.append({
        "id": cfg["id"], "arquitectura": cfg["arquitectura"], "regularizacion": cfg["regularizacion"],
        "val_accuracy": accuracy_score(y_validacion, y_pred_val),
        "val_f1_macro": f1_score(y_validacion, y_pred_val, average="macro", zero_division=0),
        "val_loss": h["val_loss"][mejor_epoca],
        "brecha_train_val": h["accuracy"][mejor_epoca] - h["val_accuracy"][mejor_epoca],
        "epocas": len(h["loss"]), "tiempo_seg": round(time.time() - inicio, 1),
        "parametros": modelo.count_params()})
    historiales[cfg["id"]] = h
    modelo.save(f"{RUTA_SALIDA}/modelos/{cfg['id']}.keras")
    print(f"[{cfg['id']}] {cfg['arquitectura']} {cfg['regularizacion']} -> val_acc={resultados[-1]['val_accuracy']:.3f} | val_f1={resultados[-1]['val_f1_macro']:.3f}")

tabla_resultados = pd.DataFrame(resultados)
tabla_resultados.to_csv(f"{RUTA_SALIDA}/resultados_12_configuraciones.csv", index=False)
tabla_resultados

## 4. Curvas de aprendizaje (Loss y Accuracy, train vs val) de las 12 configuraciones

In [ ]:
for cfg in configuraciones:
    h = historiales[cfg["id"]]
    figura, ejes = plt.subplots(1, 2, figsize=(8, 3))
    ejes[0].plot(h["loss"], label="train"); ejes[0].plot(h["val_loss"], label="val"); ejes[0].set_title("Loss")
    ejes[1].plot(h["accuracy"], label="train"); ejes[1].plot(h["val_accuracy"], label="val"); ejes[1].set_title("Accuracy")
    for eje in ejes:
        eje.set_xlabel("Época"); eje.legend()
    figura.suptitle(f"{cfg['id']} · {cfg['arquitectura']} {cfg['regularizacion']}")
    figura.tight_layout()
    figura.savefig(f"{RUTA_SALIDA}/curvas/{cfg['id']}.png", dpi=120)
    plt.show()

## 5. Selección del mejor modelo (con validación F1-macro) y evaluación cuantitativa en test

In [ ]:
mejor = tabla_resultados.sort_values(["val_f1_macro", "val_loss"], ascending=[False, True]).iloc[0]
print("Mejor configuración seleccionada:", mejor["id"], mejor["arquitectura"], mejor["regularizacion"])
modelo_final = tf.keras.models.load_model(f"{RUTA_SALIDA}/modelos/{mejor['id']}.keras")

probabilidades_prueba = modelo_final.predict(X_prueba, verbose=0)
y_pred_prueba = np.argmax(probabilidades_prueba, axis=1)
indices_clases = list(range(numero_clases))

print(f"\nAccuracy en test: {accuracy_score(y_prueba, y_pred_prueba):.3f}")
reporte = classification_report(y_prueba, y_pred_prueba, labels=indices_clases,
                                target_names=nombres_clases, zero_division=0, output_dict=True)
tabla_reporte = pd.DataFrame(reporte).T.round(2)
tabla_reporte.to_csv(f"{RUTA_SALIDA}/reporte_clasificacion_test.csv")
print(classification_report(y_prueba, y_pred_prueba, labels=indices_clases, target_names=nombres_clases, zero_division=0))

matriz_confusion = confusion_matrix(y_prueba, y_pred_prueba, labels=indices_clases)
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_confusion, annot=True, fmt="d", cmap="Blues", xticklabels=nombres_clases, yticklabels=nombres_clases)
plt.title(f"Matriz de confusión (test) — {mejor['id']}"); plt.xlabel("Predicho"); plt.ylabel("Real")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(f"{RUTA_SALIDA}/matriz_confusion_test.png", dpi=130); plt.show()

## 6. Evaluación cualitativa: aciertos y errores en test

In [ ]:
confianza_prueba = probabilidades_prueba.max(axis=1)
indices_aciertos = np.where(y_pred_prueba == y_prueba)[0][:4]
indices_errores = np.where(y_pred_prueba != y_prueba)[0][:4]

figura, ejes = plt.subplots(2, 4, figsize=(14, 7))
for fila, (indices, titulo) in enumerate([(indices_aciertos, "ACIERTO"), (indices_errores, "ERROR")]):
    for columna in range(4):
        eje = ejes[fila][columna]; eje.axis("off")
        if columna < len(indices):
            i = indices[columna]
            eje.imshow(X_prueba[i])
            eje.set_title(f"{titulo}\nreal: {nombres_clases[y_prueba[i]]}\npred: {nombres_clases[y_pred_prueba[i]]} ({confianza_prueba[i]:.2f})",
                          fontsize=9, color="green" if titulo == "ACIERTO" else "red")
plt.tight_layout(); plt.savefig(f"{RUTA_SALIDA}/aciertos_errores_test.png", dpi=130); plt.show()

## 7. Umbral de confianza ("la persona no está en la base de datos")
Si la probabilidad máxima es menor que el umbral calibrado, la API responderá que la persona no está registrada.

In [ ]:
confianza_validacion = modelo_final.predict(X_validacion, verbose=0)
prediccion_validacion = confianza_validacion.argmax(axis=1)
confianza_aciertos_val = confianza_validacion.max(axis=1)[prediccion_validacion == y_validacion]

# Si separamos fotos de compañeros no registrados en el dataset (ci 01 / compañero inf):
todas_desconocidas = list(fotos_desconocidas) if 'fotos_desconocidas' in globals() and len(fotos_desconocidas) > 0 else []

ruta_desc = globals().get('RUTA_DESCONOCIDOS_DRIVE', globals().get('RUTA_DESCONOCIDOS', ''))
if ruta_desc and os.path.isdir(ruta_desc):
    for carp, _, archivos in os.walk(ruta_desc):
        for a in archivos:
            if a.lower().endswith((".jpg", ".jpeg", ".png")):
                img = leer_imagen_rgb(os.path.join(carp, a))
                if img is not None:
                    todas_desconocidas.append(img)

if len(todas_desconocidas) > 0:
    confianza_desconocidos = modelo_final.predict(np.array(todas_desconocidas), verbose=0).max(axis=1)
    umbrales = np.arange(0.30, 1.00, 0.01)
    puntajes = [((confianza_aciertos_val >= u).mean() + (confianza_desconocidos < u).mean()) / 2 for u in umbrales]
    umbral_confianza = float(umbrales[int(np.argmax(puntajes))])
    print(f"Umbral calibrado con {len(todas_desconocidas)} fotos de personas desconocidas: {round(umbral_confianza, 2)}")
else:
    umbral_confianza = float(max(0.5, np.percentile(confianza_aciertos_val, 10)))
    print(f"Sin fotos de desconocidos: umbral por percentil 10 de aciertos en val: {round(umbral_confianza, 2)}")

print("UMBRAL FINAL CALIBRADO:", round(umbral_confianza, 2))
rechazadas_test = (confianza_prueba < umbral_confianza).mean()
print(f"Fotos de test que quedarían como 'no está en la BD': {rechazadas_test:.0%}")

## 8. Exportar modelo, metadatos y DESCARGA AUTOMÁTICA
Esta celda guarda `modelo_cnn.keras` y `clases.json`, respalda en Google Drive e **inicia la descarga directa en tu navegador**.

In [ ]:
# 1. Guardar modelo final y metadatos
modelo_final.save(f"{RUTA_SALIDA}/modelo_cnn.keras")
with open(f"{RUTA_SALIDA}/clases.json", "w", encoding="utf-8") as archivo:
    json.dump({"clases": nombres_clases, "umbral": umbral_confianza, "tamano_imagen": TAMANO_IMAGEN,
               "configuracion": f"{mejor['id']} {mejor['arquitectura']} {mejor['regularizacion']}"},
              archivo, ensure_ascii=False, indent=2)

# 2. Respaldo opcional en Google Drive
try:
    destino = "/content/drive/MyDrive/ep2_resultados"
    shutil.copytree(RUTA_SALIDA, destino, dirs_exist_ok=True)
    print("Respaldo guardado en Google Drive:", destino)
except Exception as e:
    print("Respaldo en Drive omitido o fallido:", e)

print("\n--- ¡ENTRENAMIENTO COMPLETADO! ---")
print("Iniciando descarga directa de 'modelo_cnn.keras' y 'clases.json' a tu computador...")

# 3. Descarga directa en el navegador
try:
    files.download(f"{RUTA_SALIDA}/modelo_cnn.keras")
    files.download(f"{RUTA_SALIDA}/clases.json")
except Exception as e:
    print("Nota: Si la descarga automática no inició, descarga los archivos manualmente desde la carpeta 'resultados' en el panel izquierdo de Colab:", e)

print("\nSiguiente paso: Mueve los dos archivos descargados dentro de la carpeta 'api/' de tu proyecto.")